In [1]:
#Importing relevant libraries
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits 
import os
import feets
import matplotlib as mpl
from matplotlib.colors import LogNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import seaborn as sns
from matplotlib.ticker import LogFormatter, FixedLocator
import glob
from pathlib import Path
from extinction import fitzpatrick99

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 8.0,
    "axes.linewidth": 0.6,
    "axes.labelsize": 14.0,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "xtick.major.size": 2.8,
    "ytick.major.size": 2.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.frameon": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

Defining relevant functions

In [2]:
############ Jurassic Park extinction correction #########
def jurassic_park(df, log_df):
    eff_wavelengths = {'u': 3641, 'g': 4704, 'r': 6155, 'i': 7504, 'z': 8695, 'y': 10056}

    # Merge to get EBV
    df_merged = df.merge(log_df[['object_id', 'EBV']], on='object_id', how='left')

    # Map filter to effective wavelength
    df_merged['eff_wl'] = df_merged['Filter'].map(eff_wavelengths)

    # Compute extinction
    df_merged['A_lambda'] = [
        fitzpatrick99(np.array([wl]), ebv*3.1)[0] if not np.isnan(wl) else 0
        for wl, ebv in zip(df_merged['eff_wl'], df_merged['EBV'])
    ]

    # Flux correction
    df_merged['Flux_ext'] = df_merged['Flux'] * 10**(df_merged['A_lambda'] / 2.5)

    return df_merged

########## Feature extraction ##############

def feat_extr(df, save_path, band):
    feature_list = {
    'Amplitude',
    'BeyondNStd',              
    'MedianAbsDev',            
    'MedianBRP',               
    'MaxSlope',                
    'LinearTrend',            
    'PairSlopeTrend',          
    'PercentAmplitude',        
    'PercentDiffPercentile',   
    'Skew',                    
    'SmallKurtosis',           
    'Std',                     
    'Mean',                   
    'MeanVariance',            
    'Rcs', 
    'Eta', 
    'EtaE',
    'StetsonJ',
    'StetsonK',
    'ExcessVariance',
    'Freq1_harmonics_amplitude_1',
    'Freq1_harmonics_amplitude_2',
    'Freq1_harmonics_amplitude_3',
    'Freq1_harmonics_rel_phase_1',
    'Freq1_harmonics_rel_phase_2',
    'Freq1_harmonics_rel_phase_3',
    'PeriodLS',
    'Period_fit',
    'CAR_mean',
    'CAR_tau',
    'CAR_sigma'
    }
    df_list = []
    df_band = df[df['Filter']==band]
    grouped = df_band.groupby('object_id')
    
    for idx, group in grouped:

        #Removing noise
        time, mag, error = feets.preprocess.remove_noise(
        time=group['Time (MJD)'].values,
        magnitude=group['Flux_ext'].values,
        error=group['Flux_err'].values
        )
        if len(time) < 6:
            continue

        _, unique_indices = np.unique(time, return_index=True)
        time = time[unique_indices]
        mag = mag[unique_indices]
        error = error[unique_indices]
        # Ensure time is strictly ascending
        sorted_indices = np.argsort(time)
        time = time[sorted_indices]
        mag = mag[sorted_indices]
        error = error[sorted_indices]
        example = {'time': time,
        'magnitude': mag,
        'error': error}
        fs = feets.FeatureSpace(data={"magnitude", "time", 'error'},only=feature_list)
        features = fs.extract(**example)
        feat_df = features.as_frame()
        feat_df['object_id'] = idx
        df_list.append(feat_df)
    final_df = pd.concat(df_list, ignore_index=True)

    os.makedirs(save_path, exist_ok=True)
    final_df.to_csv(os.path.join(save_path, f'feature_df_{band}.csv')) 

Combining all the splits

In [3]:
######### Paths #########
base = Path("Splits")

save_folder = 'Denoised_normalized_light_curves_QNPy'
filtered_folder = 'Filtered_data_QNPy'
os.makedirs(save_folder, exist_ok=True)
os.makedirs(filtered_folder, exist_ok=True)

######### Load train/test data #########
train_dfs, test_dfs = [], []
for split in sorted(base.glob("split_*")):
    train_path = split / "train_full_lightcurves.csv"
    test_path = split / "test_full_lightcurves.csv"
    if train_path.exists(): train_dfs.append(pd.read_csv(train_path))
    if test_path.exists(): test_dfs.append(pd.read_csv(test_path))

df_train = pd.concat(train_dfs, ignore_index=True)
df_test = pd.concat(test_dfs, ignore_index=True)
train_log = pd.read_csv('train_log.csv')
test_log = pd.read_csv('test_log.csv')

######### De-extinction #########
df_train = jurassic_park(df_train, train_log)
df_test = jurassic_park(df_test, test_log)

Starting feature extraction

In [5]:
print('Starting feature extraction')
bands = ['u', 'g', 'r', 'i', 'z', 'y']
save_path = 'Feature_dataset'
for band in bands:
    feat_extr(df_train, os.path.join(save_path, 'train'), band=band)
    feat_extr(df_test, os.path.join(save_path, 'test'), band=band)
print('Extraction finished!')

Starting feature extraction


KeyboardInterrupt: 